In [1]:
rdd = spark.sparkContext.textFile("hdfs://localhost:9000/user/ml-32m/ratings.csv")

In [2]:
rdd.take(5)

['userId,movieId,rating,timestamp',
 '1,17,4.0,944249077',
 '1,25,1.0,944250228',
 '1,29,2.0,943230976',
 '1,30,5.0,944249077']

In [3]:
def is_valid_rating(line):
    try:
        float(line[2])
        return True
    except:
        return False

In [4]:
raw_rdd = rdd.map(lambda line: line.split(","))
split_rdd = raw_rdd.filter(lambda line: len(line) == 4)
clean_rdd = split_rdd.filter(is_valid_rating)
movie_rdd = clean_rdd.map(lambda line: (int(line[1]), float(line[2])))

In [5]:
movie_rdd.first()

(17, 4.0)

In [6]:
movie_rdd_transformed = movie_rdd.map(lambda line: (line[0], (line[1], 1)))

In [7]:
movie_rdd_transformed.first()

(17, (4.0, 1))

In [8]:
movie_rdd_reduced = movie_rdd_transformed.reduceByKey(lambda record1, record2:(record1[0] + record2[0], record1[1] + record2[1])) 

In [9]:
movie_rdd_reduced = movie_rdd_reduced.filter(lambda record: record[1][1] > 100)

In [10]:
movie_rdd_average = movie_rdd_reduced.map(lambda line: (line[0], line[1][0] / line[1][1]))

In [11]:
movie_rdd_sorted = movie_rdd_average.sortBy(lambda line: line[1], False)

In [12]:
movie_rdd_sorted.take(10)

[(171011, 4.4468302658486705),
 (159817, 4.444369063772049),
 (170705, 4.426538598363572),
 (318, 4.404613860039444),
 (171495, 4.330081300813008),
 (858, 4.317030403371463),
 (202439, 4.312253641816624),
 (179135, 4.300085984522786),
 (198185, 4.298684210526316),
 (220528, 4.28619153674833)]